### Generates the source data for Figures 2 and 3

In [ ]:
"""
Spectral Power per Subject and Phase into a Single File, for Total and Aperiodic-Corrected Spectra

This notebook aggregates MEG Power Spectral Density (PSD) data for each subject and phase into a single TSV file (one for total power and one for aperiodic-corrected power). It reads individual PSD files, extracts relevant information, and compiles it into a structured format for further analysis and visualization.

output files:
- avgspectra_{megtype}{snorm}.tsv: Contains the (relative) average spectra for each subject and phase, with columns for subject ID, phase, and power values per frequency bin from 2 to 40 Hz.
- avgspectra_{megtype}_per{snorm}.tsv: Contains the (relative) aperiodic-corrected (periodic) spectra for each subject and phase, with similar columns as above.

Author: Maité Crespo García
Affiliation: MRC Cognition and Brain Sciences Unit, Cambridge, UK
Date: 2026 (last modified)
"""

from specparam.utils.spectral import interpolate_spectra
import pandas as pd
import numpy as np
import mne
import matplotlib.pyplot as plt
import os

print(os.getcwd())
datadir = os.path.abspath(os.path.join(os.path.dirname( os.getcwd() ), '.', 'data'))
print(f'Data directory: {datadir}') # Data dir of this repository (the output .tsv files are already saved in this directory, so careful if you want to run this notebook again, to avoid overwriting them)

maindir = '' # main directory where the bids repository is located, with the derivatives folder inside it.

pipver = ''
task = 'rest'
phases = ['p2', 'p5']

lfreq = 0.1 #Hz
hfreq = 145.0 #Hz
fsample = 300.0 #Hz
frange = f"{round(lfreq, 1)}-{int(hfreq)}Hz"

trans = True # Whether to use head transformation or not
zmm = 44 # destination z coordinate head position in mm

icselection = 'ecg04eog08' # 'allbutecg04' #'eog08' #
proc = 'filt' + icselection #'sss' #'clean'

cropdata = 532 
epoch_duration = 2  # in seconds
powmethod = 'WL'  # 'MT'  'WL'

psddesc = f'dur{cropdata}sepo{epoch_duration}s{powmethod}'

# ---- File with subjects and arms ----
subjlistfile = os.path.join(datadir,f'meglong_{task}_subjects.tsv')

# ---- File with subjects and age in each phase ----
agefile = os.path.join(datadir,f'meglong_rest_age.tsv')

normalize_psd = True
snorm = '_normalized' if normalize_psd else ''

megtype = 'grad' # 'grad', 'mag',

# --- File to save the average spectra for each age group and phase, to avoid having to read the psd files again if we want to change the plotting parameters ---
avgspectrafile = os.path.join(datadir, f'avgspectra_{megtype}{snorm}.tsv')

# --- File to save the aperiodic-corrected (periodic) spectra for each age group and phase, to avoid having to read the psd files again if we want to change the plotting parameters ---
avgspectrafile_per = os.path.join(datadir, f'avgspectra_{megtype}_per{snorm}.tsv')

age_groups = ['Young', 'Middle', 'Old']

psd_deriv_folder = f'mne-bids-pipeline_{pipver}_filt{frange}_fs{int(fsample)}Hz_trans_z{zmm}mm'
goodepochs_deriv_folder = psd_deriv_folder

# --- Read in the subjects and age dataframes ---
subjectsdf = pd.read_csv(subjlistfile, sep='\t', index_col=0)
agedf = pd.read_csv(agefile, sep='\t', index_col=0)
agedf.rename(columns={'p2_meg_age': 'p2_age', 'p5_meg_age': 'p5_age'}, inplace=True)

subjects = subjectsdf.index.tolist()      
agedf = agedf.loc[subjects]
subjectsdf = subjectsdf.loc[subjects]

# --- Define age groups based on age in phase 2 ---
age_bins = np.percentile(agedf['p2_age'], [0, 100/3, 2*100/3, 100])
subjectsdf['Age_group'] = pd.cut(agedf['p2_age'], bins=age_bins, labels=age_groups, include_lowest=True)
agedf.loc[subjects, 'Age_group'] = subjectsdf.loc[subjects, 'Age_group']

# Generate structural summary metrics without using loops
summary_df = agedf.groupby('Age_group', observed=True).agg(
    min_age=('p2_age', 'min'),
    max_age=('p2_age', 'max'),
    min_age_p5=('p5_age', 'min'),
    max_age_p5=('p5_age', 'max'),
    n_subjects=('p2_age', 'count')
)

# Convert directly to a dictionary matching your original schema
age_groups_dict = summary_df.to_dict(orient='index')

print(age_groups_dict)

phases_dict = {
    'p2': {'label': 'Phase 2', 'color': 'royalblue'},
    'p5': {'label': 'Phase 5', 'color': 'orange'}
}

{'Young': {'min_age': 24.3, 'max_age': 47.6, 'min_age_p5': 35.6, 'max_age_p5': 58.7, 'n_subjects': 45}, 'Middle': {'min_age': 47.8, 'max_age': 63.1, 'min_age_p5': 58.9, 'max_age_p5': 74.5, 'n_subjects': 44}, 'Old': {'min_age': 63.2, 'max_age': 84.1, 'min_age_p5': 70.9, 'max_age_p5': 95.5, 'n_subjects': 44}}


In [ ]:
fitting_param = 'finley'
aper_deriv_folder = f'aperiodic_filt{frange}_fs{int(fsample)}Hz_trans_z{zmm}mm'

# General figure settings
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Nimbus Sans"]
plt.rcParams.update({'font.size': 8}) # setting font size to 8

def get_powerunits(megtype):
    id = subjects[0]
    armx = subjectsdf.loc[id,'arm']        

    # --- Define the psd directory and file ---
    bids_project_folder = f'BIDS_long_{phase}_{task}_arm{armx}'
    psd_derivdir = os.path.join(maindir, bids_project_folder,
            'derivatives', psd_deriv_folder)
    psd_megdir = os.path.join(psd_derivdir, 'sub-'+id, 'meg')

    psdfilename = f'sub-{id}_task-{task}_proc-{proc}_desc-{psddesc}_psd.hdf5'
    psdfile = os.path.join(psd_megdir, psdfilename)

    # --- Read the power spectrum data ---
    psd = mne.time_frequency.read_spectrum(psdfile)
    chan_units = psd.units()[megtype]
    return chan_units

if normalize_psd:
    chan_units = '(normalized, a.u.)'
else:
    chan_units = get_powerunits('grad')

def load_subject_spectra(id, armx, phase):
    # --- Define the psd directory and file ---
    bids_project_folder = f'BIDS_long_{phase}_{task}_arm{armx}'
    psd_derivdir = os.path.join(maindir, bids_project_folder,
            'derivatives', psd_deriv_folder)
    psd_megdir = os.path.join(psd_derivdir, 'sub-'+id, 'meg')

    psdfilename = f'sub-{id}_task-{task}_proc-{proc}_desc-{psddesc}_psd.hdf5'
    psdfile = os.path.join(psd_megdir, psdfilename)

    # --- Define the bad epochs file ---
    goodepochs_derivdir = os.path.join(maindir, bids_project_folder,
            'derivatives', goodepochs_deriv_folder)
    goodepochs_megdir = os.path.join(goodepochs_derivdir, 'sub-'+id, 'meg')

    badepochsfilename = f'sub-{id}_task-{task}_proc-sss_desc-dur{cropdata}sepo{epoch_duration}s_badepochs.npy'
    badepochsfile = os.path.join(goodepochs_megdir, badepochsfilename)

    # --- Read the power spectrum data ---
    psd = mne.time_frequency.read_spectrum(psdfile)

    # --- Get the psds for each epoch and channel ---
    spectra, freqs = psd.pick(megtype).get_data(return_freqs=True)
    
    if normalize_psd:
        chan_units = '(normalized, a.u.)'
    else:
        chan_units = psd.units()[megtype]
    del psd # free memory

    # limit to 1-45 Hz
    freq_mask = (freqs >= 1) & (freqs <= 40)
    freqs = freqs[freq_mask]
    spectra = spectra[:, :, freq_mask]

    # --- Read the list of good epochs ---
    vardict = np.load(badepochsfile, allow_pickle=True).item()
    good_epochs = vardict['good_epochs']

    ### Remove the first 30-s of the psd, to wait until the participant has settled in. This is equivalent to removing the first 3 epochs of 10-s, or the first 15 epochs of 2-s.
    n_epochs_to_remove = int(30/epoch_duration)
    good_epochs = good_epochs[good_epochs >= n_epochs_to_remove]

    # --- Select the good epochs and average PSD across epochs ---
    spectra = spectra[good_epochs,:,:]
    spectra = np.average(spectra, axis=0)

    # ---- Interpolate 23.4 Hz (Golan's) noise ----            
    freqs, spectra = interpolate_spectra(freqs, spectra, [21.9, 23.9]) # channels x frequencies

    # --- Average PSD across channels ---
    spectra = np.average(spectra, axis=0)  # average across channels
    spectra_ori = spectra.copy()

    if normalize_psd:
        spectra = spectra / np.trapezoid(spectra, freqs)  # normalize PSD by total power

    return spectra, freqs, chan_units, spectra_ori

def load_subject_periodic_spectra(id, armx, phase, freqs, spectra_ori):
    # Load the aperiodic spectra for this subject and phase
    # --- Define the psd directory and file ---
    bids_project_folder = f'BIDS_long_{phase}_{task}_arm{armx}'
    psd_derivdir = os.path.join(maindir, bids_project_folder,
            'derivatives', aper_deriv_folder)
    psd_megdir = os.path.join(psd_derivdir, 'sub-'+id, 'meg')

    psdfilename = f'sub-{id}_task-{task}_proc-{proc}_desc-{psddesc}{megtype}{fitting_param}_specparam_aperiodic.npy'
    psdfile = os.path.join(psd_megdir, psdfilename)

    # --- Read the power spectrum data ---
    psd = np.load(psdfile, allow_pickle=True).item()
    spectra_aper = psd['spectra']  # epochs x channels x frequencies
    freqs_aper = psd['freqs']
    
    # --- Average PSD across channels ---
    spectra_aper = np.average(spectra_aper, axis=0)  # average across channels

    freq_mask = (freqs >= freqs_aper[0]) & (freqs <= freqs_aper[-1])
    spectra_ori = spectra_ori[freq_mask]
    
    spectra_per = np.log(np.divide(spectra_ori, spectra_aper))
    freqs_per = freqs_aper.copy()

    return spectra_per, freqs_per

def get_dataset_df(subject, phase, freqs, spectra):
    freqs_column_names = [f'Freq_{f:.1f}Hz' for f in freqs]

    df_temp = pd.DataFrame({
        'Subject': subject,
        'Phase': phase,
        **{col_name: spectra[i] for i, col_name in enumerate(freqs_column_names)}
    }, index=[0])  # index=[0] to create a single-row dataframe

    return df_temp
# ---- End of function definitions ----

# --- Create a dataframe to store the total spectra for each subject and phase, between 0-40 Hz.

dataframe_list = []

# Loop over subjects 
for id in subjects: # just do one subject for now to test
    armx = subjectsdf.loc[id,'arm']
    print('Processing total power subject:', id)

    # Loop over phases
    for phase in phases:                    
        print(f'  Processing phase: {phase}')

        # Load the total spectrum for this subject and phase
        spectra, freqs, _, _ = load_subject_spectra(id, armx, phase)

        df_temp = get_dataset_df(subject=id, phase=phase, freqs=freqs, spectra=spectra)

        dataframe_list.append(df_temp)

df_final = pd.concat(dataframe_list, ignore_index=True)  
df_final.to_csv(os.path.join(datadir, avgspectrafile), sep='\t', index=False)     

# --- Create a dataframe to store the aperiodic spectra for each subject and phase, between 0-40 Hz.
dataframe_list = []
# Loop over subjects 
for id in subjects: # just do one subject for now to test
    armx = subjectsdf.loc[id,'arm']
    print('Processing periodic power subject:', id)

    # Loop over phases
    for phase in phases:                    
        print(f'  Processing phase: {phase}')

        # Load the total spectrum for this subject and phase
        _, freqs, _, spectra_ori = load_subject_spectra(id, armx, phase)

        # Load the aperiodic-corrected spectrum for this subject and phase (log10 ratio of total/aperiodic)
        spectra_per, freqs_per = load_subject_periodic_spectra(id, armx, phase, freqs, spectra_ori)

        df_temp = get_dataset_df(subject=id, phase=phase, freqs=freqs_per, spectra=spectra_per)

        dataframe_list.append(df_temp)

df_final = pd.concat(dataframe_list, ignore_index=True)  
df_final.to_csv(os.path.join(datadir, avgspectrafile_per), sep='\t', index=False)     

Processing total power subject: CC120049
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC120065
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC120218
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC120470
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC120640
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC120795
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC121428
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC210023
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC210088
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC220107
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC220335
  Processing phase: p2
  Processing phase: p5
Processing total power subject: CC220419
  